In [ ]:
=====================================================================================================
Critical-porosity model-selection and profile diagnostics on Baek et al. data.
This script uses five post-baseline porosities and the published normalized permeabilities,
with final normalized permeability reconstructed from dimensional measurements.
AICc assumes iid Gaussian natural-log residuals, with a fitted error variance.

$$\text{Created by Avulundiah Edwin Phiri}$$
$$Department of Mathematics, Copperbelt University, Kitwe$$
$$Date: 24/07/2026$$

=======================================================================================================

In [1]:
"""
This assumption and the extremely small N=5 are essential qualifications.
"""
from pathlib import Path
import csv,numpy as np
from scipy.optimize import minimize_scalar,brentq
ROOT=Path(__file__).resolve().parent
r=np.array([.282,.221,.207,.176,.172])/.319
q=np.array([.6021,.1602,.0816,.0012,7.54e-13/1.23e-9])
obs=np.log(q);N=len(r)
def cp(psi):
 u=np.log((r-psi)/(1-psi));m=np.dot(u,obs)/np.dot(u,u)
 return m,float(np.sum((m*u-obs)**2))
res=minimize_scalar(lambda x:cp(x)[1],bounds=(0,min(r)-1e-8),method='bounded',options={'xatol':1e-14})
psi=res.x;m,rs_cp=cp(psi);val=np.sqrt(rs_cp/N)
lo=brentq(lambda x:cp(x)[1]-rs_cp*1.21,0,psi)
hi=brentq(lambda x:cp(x)[1]-rs_cp*1.21,psi,min(r)-1e-10)
nlog=float(np.dot(np.log(r),obs)/np.dot(np.log(r),np.log(r)))
rs_pw=float(np.sum((nlog*np.log(r)-obs)**2))
kc=r**3*((1-.319)/(1-.319*r))**2;rs_kc=float(np.sum((np.log(kc)-obs)**2))
def aicc(rss,k_mean):
 params=k_mean+1 # fitted Gaussian variance
 return N*np.log(rss/N)+2*params+2*params*(params+1)/(N-params-1)
rows=[]
for name,parameters,rss in [('Kozeny-Carman',0,rs_kc),('Log-optimal power law',1,rs_pw),('Critical porosity',2,rs_cp)]:
 rows.append(dict(model=name,mean_parameters=parameters,n_observations=N,LogRMSE=(rss/N)**.5,
                  RSS_log=rss,AICc_with_variance=aicc(rss,parameters)))
with (ROOT/'model_selection_AICc.csv').open('w',newline='') as f:
 wr=csv.DictWriter(f,fieldnames=list(rows[0]));wr.writeheader();wr.writerows(rows)
with (ROOT/'critical_porosity_profile.csv').open('w',newline='') as f:
 writer=csv.writer(f);writer.writerow(['psi','m_refitted','LogRMSE','within_10pct'])
 for pp in np.linspace(0,min(r)-1e-6,2001):
  mm,rr=cp(pp);writer.writerow([pp,mm,(rr/N)**.5,int(rr<=rs_cp*1.21)])
print('psi',psi,'m',m,'nlog',nlog,'LogRMSE',val,'10% psi range',lo,hi,'AICc',rows)


NameError: name '__file__' is not defined